In [1]:
import swiftlet
from swiftlet import Transformer, Visitor

In [2]:
swiftlet.__version__

'0.1.0'

# Math Expression parser

In [4]:

grammar = """
    start: expr
    expr: expr "+" factors -> add
        | expr "-" factors -> sub
        | factors

    factors: factors "*" INT -> mul
        | factors "/" INT -> div
        | INT

    %import (WS, INT)
    %ignore WS
    """

text = "10 * 2 - 5 * 1 + 8 / 2 - 8"

parser = swiftlet.Swiftlet(grammar, debug=True)
parsed = parser.parse(text)


AST of Grammar
start
  rule
    non_terminal  start
    or_expansion
      non_terminal  expr
  rule
    non_terminal  expr
    or_expansion
      or_expansion
        or_expansion
          alias
            non_terminal  expr
            string  "+"
            non_terminal  factors
            non_terminal  add
        alias
          non_terminal  expr
          string  "-"
          non_terminal  factors
          non_terminal  sub
      non_terminal  factors
  rule
    non_terminal  factors
    or_expansion
      or_expansion
        or_expansion
          alias
            non_terminal  factors
            string  "*"
            terminal  INT
            non_terminal  mul
        alias
          non_terminal  factors
          string  "/"
          terminal  INT
          non_terminal  div
      terminal  INT
  import
    name_list
      name_list
        terminal  WS
      terminal  INT
  ignore
    or_expansion
      terminal  WS


Terminals
TerminalDef { name: Terminal(WS),

In [5]:
parsed.pretty_print()

 start
   sub
     add
       sub
         expr
           mul
             factors   10
             *
             2
         -
         mul
           factors   5
           *
           1
       +
       div
         factors   8
         /
         2
     -
     factors   8


In [6]:

class Solve(Transformer):
    def start(self, children):
        return children[0]

    def expr(self, children):
        return children[0]

    def factors(self, children):
        return children[0]

    def add(self, children):
        return children[0] + children[2]

    def sub(self, children):
        return children[0] - children[2]

    def mul(self, children):
        return children[0] * children[2]

    def div(self, children):
        return children[0] / children[2]

Solve()(parsed)

11.0

In [7]:
eval(text)

11.0

In [8]:
from abc import ABC
from swiftlet import Tree, Token, ExceptionTreeType


In [9]:
class AstVisitor(Visitor):
    def expr(self, tree):
        tree.set_children(["number"])

    def number(self, tree):
        print("-> ", tree)
        tree.set_children(["number"])

visitor = AstVisitor()
parsed.pretty_print()
visitor(parsed)

 start
   expr
     expr
       number   12.34
       +
       number   10
     -
     number   0.23
_user_func = 'start'
'AstVisitor' object has no attribute 'start'
_user_func = 'expr'


In [10]:
parsed.pretty_print()

 start
   expr   number
